# Análise de características usando um Multilayer Perceptron

> **Autores:** Alexandre Maciel e Vinicius de Lima.

Esse notebook calcula o F1-score do K-NN para diferentes tamanhos para as bases geradas por [bases.ipynb](./bases.ipynb), com as caractéristicas extraídas das raças de cachorro Samoieda e Pug e das raças de gato Russian Blue e Birman.

In [4]:
import pandas as pd
from itertools import product
from typing import Literal

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split

Nós temos 4 raças de cachorro no total, portanto 4 classes $(N_o = 4)$, além disso a melhor base foi `PCA_075_HOG_256_32x32`, que possui 83 atributos $(N_i = 83)$. Então, a 
quantidade de neurônios a partir da regra geométrica $\lfloor\sqrt{N_i \cdot N_o}\rfloor$, teremos 18 neurônios na camada escondida.

$$
\lfloor\sqrt{N_i \cdot N_o}\rfloor = 18
$$

Aplicando a mesma regra para a regra da média, teremos:

$$
\lfloor\frac{2}{3} N_i + N_o \rfloor = 59
$$

In [ ]:
from sklearn.metrics import f1_score


df = pd.read_csv("bases/PCA_075_HOG_256_32x32.csv")
xs = df.iloc[:, :-1]
ys = df["label"]

xs_train, xs_test, ys_train, ys_test = train_test_split(xs, ys, random_state=42)

activation = "relu"
hidden_layer_sizes = [18, 59]
solvers: list[Literal["adam", "sgd"]] = ["adam", "sgd"]
learning_rates = [0.0001, 0.001, 0.01, 0.1]
number_of_iterations = [1000, 1500, 2000, 2500, 3000]

results = []

for hl, s, ln, its in product(
    hidden_layer_sizes, solvers, learning_rates, number_of_iterations
):
    clf = MLPClassifier(
        hidden_layer_sizes=hl,
        activation=activation,
        solver=s,
        learning_rate_init=ln,
        max_iter=its,
    )
    _ = clf.fit(xs_train, ys_train)
    f1 = f1_score(ys_test, clf.predict(xs_test), average="weighted")
    results.append(
        {
            "hidden_layer_sizes": hl,
            "solver": s,
            "learning_rate_init": ln,
            "max_iter": its,
            "activation": activation,
            "f1_weighted": f1,
        }
    )

results_df = pd.DataFrame(results)
results_df.to_csv("mlp/results.csv", index=False)

/home/vinicius/dev/uf/img-ft-extraction-ml/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/vinicius/dev/uf/img-ft-extraction-ml/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/vinicius/dev/uf/img-ft-extraction-ml/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/vinicius/dev/uf/img-ft-extraction-ml/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10

In [8]:
sorted = results_df.nlargest(10, "f1_weighted")
sorted

,hidden_layer_sizes,solver,learning_rate_init,max_iter,activation,f1_weighted
43,59,adam,0.0001,2500,relu,0.764546
59,59,adam,0.1000,3000,relu,0.749624
25,18,sgd,0.0010,1000,relu,0.740000
52,59,adam,0.0100,2000,relu,0.740000
70,59,sgd,0.0100,1000,relu,0.734967
56,59,adam,0.1000,1500,relu,0.730027
60,59,sgd,0.0001,1000,relu,0.729784
71,59,sgd,0.0100,1500,relu,0.724855
18,18,adam,0.1000,2500,relu,0.724690
48,59,adam,0.0010,2500,relu,0.720028


In [9]:
sorted.to_csv("mlp/sorted.csv", index=False)